# 03. e + Xe⁺, расширенная модель: 6d, 7s, 8s, 4f, 7p и переходы между метастабилями

Нужна для:
* линий **класса C** (верхние уровни 6d, 7s, 4f, (¹D₂)6d: 405.7, 418.0, 420.8, 423.8, 439.3,
  482.3, 486.2, 508.1, 531.4 нм и др.);
* **каскадов** в 6p (6d, 7s → 6p) — вклад в сечения линий класса A;
* **переходов метастабиль ↔ метастабиль** и из метастабилей в 6s/5d (у Ванга их нет).

Модель `xe2_ext` (`models.py`): чётные 5s5p⁶, 5p⁴5d, 6s, 7s, 6d (+ корреляционная 7d),
нечётные 5p⁵, 5p⁴6p, 7p, 4f; состояния до 19.2 эВ. Это примерно вдвое больше
состояний, чем у Ванга: матрицы ~5×, время ~10× (начните с JMAX = 0 и стресс-теста).

Перед рассеянием: энергии мишени против NIST и A (переходы 6d/7s → 6p против NIST).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # эта папка: nbtools.py, run_model.py, models.py
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
SPECTRUM, SHEET = DATA / "A25_second.xlsx", "data2"   # измеренный спектр
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST (nist_*.csv) и список линий
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import crm, reference
import nbtools as nb
%matplotlib inline

In [ ]:
JMAX = 0            # 0 (проверка), затем 10, 25, 50
job = nb.Job(RUNS / "xe2_ext", "xe2_ext", stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=40, de=0.0136, multipoles="E1")
job.start()

In [ ]:
job.status()

In [ ]:
W = RUNS / "xe2_ext"
print((W / "target_table.txt").read_text())

## Стресс-тест после J = 0

In [ ]:
import subprocess
r = subprocess.run([sys.executable, str(REPO / "tools" / "stress_test.py"), "--target", str(W / "target"),
                    "--scat", str(W / "scat"), "--skip-matrices"], capture_output=True, text=True)
print(r.stdout[-5000:] or r.stderr[-3000:])

## Сравнение с Ванга для общих переходов

In [ ]:
ref = reference.read_xlsx(XLSX)
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz")
tg = db.Target.load(W / "target")
st = {s.name: s for s in tg.states}
# Wang number of a state: by its NIST energy
elv = [lv.energy_cm for lv in ref.levels]
wang_no = {n: nb.level_number(elv, s.exp_energy_cm) for n, s in st.items() if s.exp_energy_cm is not None}
name_of = {v: k for k, v in wang_no.items() if v and k in cs.names}
trs = [(i, j) for (i, j) in ref.sigma if i in name_of and j in name_of][:9]
fig, axs = plt.subplots(3, 3, figsize=(13, 9))
for ax, (i, j) in zip(axs.flat, trs):
    E, s = ref.sigma[(i, j)]
    ax.plot(E, s / 1e-16, lw=0.8, label="Wang 2019")
    ax.plot(cs.incident_energy(name_of[i]), cs.sigma(name_of[i], name_of[j]) / 1e-16, lw=0.8, label=f"xe2_ext, J ≤ {JMAX}")
    ax.set_xscale("log"); ax.set_title(f"{i}→{j} {ref.label(j)}", fontsize=8)
axs.flat[0].legend(fontsize=7); fig.tight_layout(); plt.show()

## Сечения линий класса C и каскады

In [ ]:
from pydbsr.transitions import TransitionTable
trE1 = nb.read_csv(W / "transitions_E1.csv")
Atab = {}
for r in trE1:
    Atab.setdefault(r["upper"], {})[r["lower"]] = Atab.get(r["upper"], {}).get(r["lower"], 0) + float(r["A_exp"])
by_e = {round(s.exp_energy_cm, 1): n for n, s in st.items() if s.exp_energy_cm is not None}
def state_at(e):
    k = min(by_e, key=lambda x: abs(x - e)); return by_e[k] if abs(k - e) < 0.5 else None
lines = [r for r in nb.read_csv(NIST / "lines.csv") if r["ion"] == "Xe II" and r["class"] == "C" and r.get("observed") == "да"]
g = cs.names.index(tg.ground.name) if tg.ground.name in cs.names else 0
rows = []
for r in lines:
    u, l = state_at(float(r["Ek_cm"])), state_at(float(r["Ei_cm"]))
    if not u or not l or u not in cs.names or u not in Atab:
        continue
    b = Atab[u].get(l, 0) / sum(Atab[u].values())
    k = [b * cs.rate(cs.names[g], u, t * 11604.5)[0] for t in (1.0, 2.0, 5.0)]
    rows.append((float(r["wl_nm"]), r["upper"], b, *k))
for x in sorted(rows):
    print(f"{x[0]:8.3f}  {x[1]:28s} BR = {x[2]:.3f}  k_line(1, 2, 5 эВ) = {x[3]:.2e} {x[4]:.2e} {x[5]:.2e}")

Данные для CRM (все пары, A): `crm_xe2_ext.json` — пишет стадия `crm`.